In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"

%load_ext autoreload
%autoreload 2

In [2]:
import ipdb
from Data import *
from address import *
from modelGen import *
from utils import cf_eval, cleanup_gpu
from models import CausalCondLatentCF, latentCFpp, PrototypeLatentCF
from models import CondLatentCF # Note: get the vanilla CondLatentCF
import numpy as np
import random 
from tensorflow import random as tf_random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product
import argparse
import os
import pandas as pd


In [3]:
SEED = 23

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf_random.set_seed(SEED)


CF_METHODS = {  
    "LatentCFpp": latentCFpp,
    "Prototype": PrototypeLatentCF,
    "CausalCACTUS": CausalCondLatentCF,
    "CACTUS": CondLatentCF
}

In [4]:
# Experimentos que a realizar
# TODO: modify the name
EXP = [
    {
        "name": "CACTUS1",
        "data": "GIVECREDIT",
        "classifier": "./exp/GIVECREDIT_class/config.json",
        "AE": "./exp/GIVECREDIT_CACTUS/config.json",
        "CFmethod": "CACTUS",
        "context": [
            "ageAbove50",
            "Dependents"
        ],
        "epochs": 100,
        "target_prob": 0.5,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.01,
        "gamma": 0.1,
        "beta": 0.01,
        "dynamicAlpha": False
    },
    {
        "name": "CACTUS2",
        "data": "GIVECREDIT",
        "classifier": "./exp/GIVECREDIT_class/config.json",
        "AE": "./exp/GIVECREDIT_CACTUS/config.json",
        "CFmethod": "CACTUS",
        "context": [
            "ageAbove50",
            "Dependents"
        ],
        "epochs": 100,
        "target_prob": 0.5,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.25,
        "gamma": 0.1,
        "beta": 0.01,
        "dynamicAlpha": False
    },
    {
        "name": "CACTUS3",
        "data": "GIVECREDIT",
        "classifier": "./exp/GIVECREDIT_class/config.json",
        "AE": "./exp/GIVECREDIT_CACTUS/config.json",
        "CFmethod": "CACTUS",
        "context": [
            "ageAbove50",
            "Dependents"
        ],
        "epochs": 100,
        "target_prob": 0.5,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.5,
        "gamma": 0.1,
        "beta": 0.01,
        "dynamicAlpha": False
    },
    {
        "name": "CACTUS4",
        "data": "GIVECREDIT",
        "classifier": "./exp/GIVECREDIT_class/config.json",
        "AE": "./exp/GIVECREDIT_CACTUS/config.json",
        "CFmethod": "CACTUS",
        "context": [
            "ageAbove50",
            "Dependents"
        ],
        "epochs": 100,
        "target_prob": 0.5,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.75,
        "gamma": 0.1,
        "beta": 0.01,
        "dynamicAlpha": False
    },
    {
        "name": "CACTUS5",
        "data": "GIVECREDIT",
        "classifier": "./exp/GIVECREDIT_class/config.json",
        "AE": "./exp/GIVECREDIT_CACTUS/config.json",
        "CFmethod": "CACTUS",
        "context": [
            "ageAbove50",
            "Dependents"
        ],
        "epochs": 100,
        "target_prob": 0.5,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.99,
        "gamma": 0.1,
        "beta": 0.01,
        "dynamicAlpha": False
    }
]

In [5]:
import re
from collections import defaultdict
# TODO: make a function to convert the causal paths for all the test samples
def build_causal_index_map(input_features, graph_str):
    """
    Returns:
        child_to_parents: dict {child_idx: [parent_idx, ...]}
        parent_to_children: dict {parent_idx: [child_idx, ...]}
        feat2idx: dict {feature_name: index}
    """

    # Feature name to index
    feat2idx = {f: i for i, f in enumerate(input_features)}

    # Extract edges using regex
    edges = re.findall(r'(\w+)\s*->\s*(\w+)', graph_str)

    child_to_parents = defaultdict(list)
    parent_to_children = defaultdict(list)

    for parent, child in edges:

        if parent not in feat2idx or child not in feat2idx:
            raise ValueError(f"Feature in graph not found in dataset: {parent} or {child}")

        p_idx = feat2idx[parent]
        c_idx = feat2idx[child]

        child_to_parents[c_idx].append(p_idx)
        parent_to_children[p_idx].append(c_idx)

    return dict(child_to_parents), dict(parent_to_children), feat2idx

# credit_rex5_constraints_min.dot from downsampled
graph_str1 = """
strict digraph {
NumberRealEstateLoansOrLines;
NumberOfTimes90DaysLate;
NumberOfDependents;
age;
DebtRatio;
NumberOfTime3059DaysPastDueNotWorse;
NumberOfOpenCreditLinesAndLoans;
RevolvingUtilizationOfUnsecuredLines;
MonthlyIncome;
NumberOfTime6089DaysPastDueNotWorse;
NumberOfTimes90DaysLate -> NumberOfOpenCreditLinesAndLoans;
NumberOfTimes90DaysLate -> NumberRealEstateLoansOrLines;
NumberOfTimes90DaysLate -> RevolvingUtilizationOfUnsecuredLines;
NumberOfTimes90DaysLate -> MonthlyIncome;
NumberOfTime3059DaysPastDueNotWorse -> NumberOfOpenCreditLinesAndLoans;
NumberOfTime3059DaysPastDueNotWorse -> NumberRealEstateLoansOrLines;
NumberOfTime3059DaysPastDueNotWorse -> MonthlyIncome;
NumberOfOpenCreditLinesAndLoans -> DebtRatio;
RevolvingUtilizationOfUnsecuredLines -> NumberOfOpenCreditLinesAndLoans;
MonthlyIncome -> NumberOfOpenCreditLinesAndLoans;
}
"""

# credit_rex6_constraints_mod.dot from downsampled
graph_str2 = """ 
strict digraph {
NumberRealEstateLoansOrLines;
NumberOfTimes90DaysLate;
NumberOfDependents;
age;
DebtRatio;
NumberOfTime3059DaysPastDueNotWorse;
NumberOfOpenCreditLinesAndLoans;
RevolvingUtilizationOfUnsecuredLines;
MonthlyIncome;
NumberOfTime6089DaysPastDueNotWorse;
NumberRealEstateLoansOrLines -> DebtRatio;
NumberOfOpenCreditLinesAndLoans -> DebtRatio;
RevolvingUtilizationOfUnsecuredLines -> NumberOfOpenCreditLinesAndLoans;
MonthlyIncome -> NumberOfOpenCreditLinesAndLoans;
}
"""

In [6]:
# # CF generation
# child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
#     data.features_lbls,
#     # graph_str
#     graph_str1
# )
# print("Feature → index:")
# print(feat2idx)
# print("\nChild → Parents (index form):")
# print(child_to_parents_dict)

## Graph 1 (Minimum priors)

In [19]:
N = 50 
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [12, 23, 34, 45, 56]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str1
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if "CausalCACTUS" in exp["name"]:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              
      
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics1 = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics1 = df_metrics1.round(3)
print(df_metrics1)





********************************************************************************************************************************************************************************************************
Running exp: CACTUS1 (0/5)
********************************************************************************************************************************************************************************************************



Reading data




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Class Good Standing: 8334.0 samples
Class Default Risk: 8334.0 samples
Age above 50 att: 7093
Dependents more than one: 8188
----------------------------------------------------------------------------------------------------




Building model
Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mai

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


restoring weights of model DNN: GIVECREDIT_class
Building model
Building model: CACTUS_VAE_tabular [gen]


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 8)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1476        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 8)            1480        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

...

2/2 [==============================] - 0s 3ms/step


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(


2/2 [==============================] - 0s 3ms/step


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(


       Model     Dataset CFMethod                Metric  Value
0    CACTUS1  GIVECREDIT   CACTUS             proximity  0.520
1    CACTUS1  GIVECREDIT   CACTUS           n_proximity  0.184
2    CACTUS1  GIVECREDIT   CACTUS              validity  0.060
3    CACTUS1  GIVECREDIT   CACTUS           compactness  0.455
4    CACTUS1  GIVECREDIT   CACTUS           lof_context  1.124
..       ...         ...      ...                   ...    ...
220  CACTUS5  GIVECREDIT   CACTUS           lof_context  1.189
221  CACTUS5  GIVECREDIT   CACTUS  causal_validity_hard  0.380
222  CACTUS5  GIVECREDIT   CACTUS  causal_validity_soft  0.784
223  CACTUS5  GIVECREDIT   CACTUS    causal_compact_val  0.335
224  CACTUS5  GIVECREDIT   CACTUS       inlier_fraction  0.866

[225 rows x 5 columns]


In [20]:
df_metrics1.to_csv("ablation-study1-vanilla-cactus-graph1.csv")

In [8]:
child_to_parents_dict

{4: [5, 1, 0, 3], 6: [5, 1], 0: [5], 3: [5, 1], 2: [4]}

In [9]:
# Pivoting the dataframe
df_metrics_pivot1 = df_metrics1.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot1.columns = df_metrics_pivot1.columns.swaplevel(0, 1)
df_metrics_pivot1 = df_metrics_pivot1[desired_order]

df_metrics_pivot1.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot1.columns = df_metrics_pivot1.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted1 = df_metrics_pivot1.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted1

Metric                 Validity          LOF  Compactness    Proximity  \
Dataset    Model                                                         
GIVECREDIT CACTUS1  0.12 ± 0.03  1.14 ± 0.04  0.52 ± 0.05  0.16 ± 0.03   
           CACTUS2  0.29 ± 0.04  1.15 ± 0.03  0.29 ± 0.04  0.26 ± 0.02   
           CACTUS3  0.54 ± 0.09  1.14 ± 0.01  0.18 ± 0.03  0.36 ± 0.03   
           CACTUS4  0.71 ± 0.08  1.16 ± 0.02  0.09 ± 0.02  0.53 ± 0.04   
           CACTUS5  0.79 ± 0.05  1.22 ± 0.03  0.08 ± 0.02  0.69 ± 0.02   

Metric             Causal Validity (Hard) Causal Validity (Soft)  \
Dataset    Model                                                   
GIVECREDIT CACTUS1            0.52 ± 0.06            0.84 ± 0.02   
           CACTUS2             0.4 ± 0.04             0.8 ± 0.03   
           CACTUS3            0.34 ± 0.04            0.78 ± 0.02   
           CACTUS4            0.34 ± 0.03            0.77 ± 0.02   
           CACTUS5            0.37 ± 0.04            0.78 ± 0.02   

Metric             Causal Compact Validity inlier_fraction  
Dataset    Model                                            
GIVECREDIT CACTUS1             0.69 ± 0.04     0.95 ± 0.02  
           CACTUS2             0.56 ± 0.03     0.95 ± 0.03  
           CACTUS3             0.48 ± 0.03     0.95 ± 0.02  
           CACTUS4             0.38 ± 0.01     0.94 ± 0.04  
           CACTUS5             0.34 ± 0.03     0.85 ± 0.05

## Graph 2 (Moderate priors)

In [10]:
N = 50 
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [11, 22, 33, 44, 55]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str2
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if "CausalCACTUS" in exp["name"]:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              
      
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics2 = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics2 = df_metrics2.round(3)
print(df_metrics2)





********************************************************************************************************************************************************************************************************
Running exp: CACTUS1 (0/5)
********************************************************************************************************************************************************************************************************



Reading data




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Class Good Standing: 8334.0 samples
Class Default Risk: 8334.0 samples
Age above 50 att: 7107
Dependents more than one: 8235
----------------------------------------------------------------------------------------------------




Building model
Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mai

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Building model
Building model: CACTUS_VAE_tabular [gen]


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 8)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1476        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 8)            1480        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

...

2/2 [==============================] - 0s 3ms/step


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(


2/2 [==============================] - 0s 3ms/step


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but QuantileTransformer was fitted without feature names
  warnings.warn(


       Model     Dataset CFMethod                Metric  Value
0    CACTUS1  GIVECREDIT   CACTUS             proximity  0.463
1    CACTUS1  GIVECREDIT   CACTUS           n_proximity  0.164
2    CACTUS1  GIVECREDIT   CACTUS              validity  0.160
3    CACTUS1  GIVECREDIT   CACTUS           compactness  0.545
4    CACTUS1  GIVECREDIT   CACTUS           lof_context  1.118
..       ...         ...      ...                   ...    ...
220  CACTUS5  GIVECREDIT   CACTUS           lof_context  1.141
221  CACTUS5  GIVECREDIT   CACTUS  causal_validity_hard  0.880
222  CACTUS5  GIVECREDIT   CACTUS  causal_validity_soft  0.940
223  CACTUS5  GIVECREDIT   CACTUS    causal_compact_val  0.402
224  CACTUS5  GIVECREDIT   CACTUS       inlier_fraction  0.944

[225 rows x 5 columns]


In [18]:
df_metrics2.to_csv("ablation-study1-vanilla-cactus-graph2.csv")

In [11]:
child_to_parents_dict

{2: [6, 4], 4: [0, 3]}

In [14]:
# Pivoting the dataframe
df_metrics_pivot2 = df_metrics2.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot2.columns = df_metrics_pivot2.columns.swaplevel(0, 1)
df_metrics_pivot2 = df_metrics_pivot2[desired_order]

df_metrics_pivot2.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot2.columns = df_metrics_pivot2.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted2 = df_metrics_pivot2.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted2

Metric                 Validity          LOF  Compactness    Proximity  \
Dataset    Model                                                         
GIVECREDIT CACTUS1  0.13 ± 0.04  1.11 ± 0.02  0.52 ± 0.06  0.16 ± 0.02   
           CACTUS2  0.23 ± 0.04  1.13 ± 0.03  0.29 ± 0.06  0.28 ± 0.04   
           CACTUS3  0.45 ± 0.08  1.13 ± 0.02  0.15 ± 0.06   0.4 ± 0.03   
           CACTUS4  0.62 ± 0.07  1.14 ± 0.03  0.08 ± 0.02  0.54 ± 0.05   
           CACTUS5  0.69 ± 0.08  1.17 ± 0.03  0.05 ± 0.03  0.66 ± 0.07   

Metric             Causal Validity (Hard) Causal Validity (Soft)  \
Dataset    Model                                                   
GIVECREDIT CACTUS1             0.9 ± 0.05            0.95 ± 0.03   
           CACTUS2            0.85 ± 0.04            0.92 ± 0.02   
           CACTUS3            0.84 ± 0.03            0.92 ± 0.01   
           CACTUS4            0.88 ± 0.03            0.94 ± 0.02   
           CACTUS5            0.88 ± 0.03            0.94 ± 0.01   

Metric             Causal Compact Validity inlier_fraction  
Dataset    Model                                            
GIVECREDIT CACTUS1             0.76 ± 0.05     0.98 ± 0.01  
           CACTUS2             0.62 ± 0.03     0.96 ± 0.02  
           CACTUS3             0.52 ± 0.03     0.97 ± 0.02  
           CACTUS4             0.46 ± 0.03     0.97 ± 0.03  
           CACTUS5             0.41 ± 0.03     0.91 ± 0.06